# RAMEC — Revisión Automática Multimodal de Entregables de Construcción

Notebook de **ejecución en Google Colab**: clona el repositorio, instala las dependencias,
descarga los pesos del modelo y ejecuta las validaciones.

**Antes de empezar:**

(Opcional) Activa GPU en *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
   La inferencia también funciona en CPU.

## 1. Configuración

In [1]:
REPO = "BRIDERI/Ramec"
TAG  = "v1.0"              # etiqueta del Release que contiene los pesos best.pt

## 2. (Opcional) Verificar GPU

In [2]:
!nvidia-smi || echo "Sin GPU: se usará CPU (la inferencia funciona igual)."

/bin/bash: line 1: nvidia-smi: command not found
Sin GPU: se usará CPU (la inferencia funciona igual).


## 3. Clonar el repositorio

In [3]:
%cd /content
!rm -rf /content/ramec
!git clone https://github.com/{REPO}.git /content/ramec
%cd /content/ramec
!grep -n "Firma_Elaboró" src/report.py
!grep -n "analizar_firmas_control" src/extract.py
!grep -n "VAL_CTRL" src/infer.py

/content
Cloning into '/content/ramec'...
remote: Enumerating objects: 103, done.
remote: Counting objects: 100% (103/103), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 103 (delta 48), reused 32 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (103/103), 87.54 KiB | 2.13 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/ramec
158:        "Firma_Elaboró": _sn(firmas[0]),
186:                "Firma_Elaboró", "Firma_Revisó", "Firma_Aprobó_1", "Firma_Aprobó_2",
674:def analizar_firmas_control(crop, filas_esperadas=None):
40:VAL_CTRL = C.NAME_TO_ID["validacion_profesional_hoja_control"]  # 9
161:        if VAL_CTRL in boxes:
162:            firma_crop = EX.crop_box(img, boxes[VAL_CTRL][0])


## 4. Dependencias del sistema (Tesseract + Poppler)
No se instalan con pip; el idioma `spa` es necesario para leer texto en español.

In [4]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-spa poppler-utils

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
(Reading database ... 118243 files and directories currently installed.)
Preparing to unpack .../libpoppler-private-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-private-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler-dev_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler-dev:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Preparing to unpack .../libpoppler118_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking libpoppler118:amd64 (22.02.0-2ubuntu0.13) over (22.02.0-2ubuntu0.12) ...
Selecting previously unselected package poppler-utils.
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.13_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.13) ...
Selecting previously unselected package tesseract-ocr-spa.
Preparing to unpa

## 5. Dependencias de Python

In [5]:
!pip -q install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 77.2 MB/s eta 0:00:00


In [9]:
!pip -q install pymupdf

## 6. Descargar los pesos del modelo (desde el Release)
Deja `models/planos/best.pt` y `models/documentos/best.pt`.

In [6]:
!REPO={REPO} TAG={TAG} bash scripts/download_models.sh

Descargando pesos desde https://github.com/BRIDERI/Ramec/releases/download/v1.0 ...
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.8M  100 38.8M    0     0  28.7M      0  0:00:01  0:00:01 --:--:-- 83.2M
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 38.7M  100 38.7M    0     0  24.9M      0  0:00:01  0:00:01 --:--:-- 41.7M
Listo. Pesos descargados:
-rw-r--r-- 1 root root 39M Jul 10 12:52 models/documentos/best.pt
-rw-r--r-- 1 root root 39M Jul 10 12:52 models/planos/best.pt


## 7. Subir los PDFs a revisar
Sube uno o más PDFs (planos y/o documentos). Se guardan en la carpeta `pdfs/`.

In [7]:
import os
os.makedirs("pdfs", exist_ok=True)
from google.colab import files
print("Selecciona uno o más PDFs...")
subidos = files.upload()
for nombre in subidos:
    os.replace(nombre, os.path.join("pdfs", nombre))
!ls -lh pdfs

Selecciona uno o más PDFs...


Saving AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf to AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf
total 27M
-rw-r--r-- 1 root root 27M Jul 10 12:54 AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf


*Alternativa:* en vez de subir archivos, puedes montar tu Google Drive y apuntar a una
carpeta con PDFs (descomenta y ajusta la ruta).

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p pdfs && cp /content/drive/MyDrive/ruta/a/tus/pdfs/*.pdf pdfs/

## 8. Ejecutar la validación

In [8]:
!python src/infer.py --pdfs pdfs --salida outputs/Reporte_validacion.xlsx

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Listo: outputs/Reporte_validacion.xlsx
  planos=0 documentos=1 validacion_profesional=1 filas estandar=1


## 9. Ver y descargar el reporte
Se genera un Excel con seis hojas de verificación.

In [ ]:
import pandas as pd
xls = pd.ExcelFile("outputs/Reporte_validacion.xlsx")
print("Hojas:", xls.sheet_names)
display(pd.read_excel(xls, "VALIDACION_PROFESIONAL"))

from google.colab import files
files.download("outputs/Reporte_validacion.xlsx")

Hojas: ['ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL']


,Ruta,Archivo,Tipo,Elaboró,Revisó,Aprobó_1,Aprobó_2,Firma_Elaboró,Firma_Revisó,Firma_Aprobó_1,Firma_Aprobó_2,Fecha_validacion,Responsables_completos,Firmas_completas,Fecha_coincide,Validacion_profesional
0,pdfs/AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,AVP-SAV-3000-D-EST-TRA-ING-000001.pdf,DOCUMENTO,Rosendo Torrens Pujadas,Eugenia Marina Acero Carrión,Armando González González,Miguel ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI
1,pdfs/AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,AVP-SAV-3000-D-MDE-SGA-ING-000001.pdf,DOCUMENTO,Eliseo Palomares,Mayra Gómez Sandoval,Armando González González,Miguel Ángel Ordóñez Gutiérrez,SI,SI,SI,SI,12/06/2026,SI,SI,SI,SI


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## (Opcional) Reentrenar los modelos
Requiere el **dataset anotado en CVAT** en `data/raw/` (no incluido en el repo por
confidencialidad/tamaño). Con tu dataset disponible, descomenta:

In [ ]:
# !python src/convert.py --planos data/raw/planos --documentos data/raw/documentos --val-frac 0.15
# !python src/train.py --task both     # genera models/<task>/best.pt

## 8.1 Validación complementaria de sellos, firmas y logos por página

In [17]:
%cd /content/ramec

print("Archivos actuales en src:")
!ls -lh src

print("\nFunciones o scripts existentes relacionados:")
!find src -maxdepth 1 -type f | sort

/content/ramec
Archivos actuales en src:
total 120K
-rw-r--r-- 1 root root 6.2K Jul 10 12:51 convert.py
-rw-r--r-- 1 root root  25K Jul 10 12:51 extract.py
-rw-r--r-- 1 root root 4.6K Jul 10 13:53 fix_excel_final.py
-rw-r--r-- 1 root root  11K Jul 10 12:51 infer.py
-rw-r--r-- 1 root root 4.8K Jul 10 12:51 nomenclatura.py
drwxr-xr-x 2 root root 4.0K Jul 10 12:54 __pycache__
-rw-r--r-- 1 root root  18K Jul 10 13:25 reporte_final.py
-rw-r--r-- 1 root root 8.0K Jul 10 12:51 report.py
-rw-r--r-- 1 root root 4.5K Jul 10 12:51 train.py
-rw-r--r-- 1 root root  15K Jul 10 13:05 validacion_paginas.py

Funciones o scripts existentes relacionados:
src/convert.py
src/extract.py
src/fix_excel_final.py
src/infer.py
src/nomenclatura.py
src/reporte_final.py
src/report.py
src/train.py
src/validacion_paginas.py


In [19]:
%cd /content/ramec

!python src/validacion_paginas.py \
  --pdfs pdfs \
  --excel outputs/Reporte_validacion.xlsx \
  --modelo models/documentos/best.pt \
  --max-pages 10

/content/ramec
Procesando páginas: AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf
Listo. Hojas técnicas creadas:
- VALIDACION_FINAL_PAGINAS
- DETALLE_SELLOS_PAGINAS
- DETALLE_LOGOS_PAGINAS


## 8.2 Generar Excel final resumido

In [20]:
!python src/reporte_final.py \
  --excel outputs/Reporte_validacion.xlsx \
  --entidad-esperada "PROINVERSIÓN | Sociedad Concesionaria Anillo Vial"

Listo. Excel final generado:
- GENERAL
- VALIDACION_PAGINAS
- VALIDACION_LOGOS


In [21]:
%%writefile src/fix_excel_final.py
from pathlib import Path
import argparse
import re
import unicodedata
from openpyxl import load_workbook
from openpyxl.styles import PatternFill


def limpiar(x):
    if x is None:
        return ""
    x = str(x).replace("\n", " ")
    x = re.sub(r"\s+", " ", x).strip()
    if x.lower() in ["nan", "none", "null"]:
        return ""
    return x


def normalizar(x):
    x = limpiar(x)
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = x.upper()
    x = re.sub(r"[^A-Z0-9 ]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def estado_ok_no(x):
    v = normalizar(x)
    if v in ["SI", "SÍ", "OK", "CONFORME", "CUMPLE", "TRUE"]:
        return "OK"
    if v in ["NO", "OBSERVADO", "NO CUMPLE", "FALSE"]:
        return "NO"
    return ""


def buscar_columna(ws, palabras):
    """
    Busca una columna por palabras clave en la fila 1.
    """
    for cell in ws[1]:
        h = normalizar(cell.value)
        if all(p in h for p in palabras):
            return cell.column
    return None


def calcular_validacion_profesional(wb):
    if "VALIDACION_PROFESIONAL" not in wb.sheetnames:
        return {}

    ws = wb["VALIDACION_PROFESIONAL"]

    col_archivo = buscar_columna(ws, ["ARCHIVO"])
    col_cumple = buscar_columna(ws, ["CUMPLE"])

    col_resp = buscar_columna(ws, ["RESPONSABLES", "COMPLETOS"])
    col_firmas = buscar_columna(ws, ["FIRMAS", "COMPLETAS"])
    col_fecha = buscar_columna(ws, ["FECHA", "COINCIDE"])

    resultados = {}

    if not col_archivo:
        return resultados

    for row in range(2, ws.max_row + 1):
        archivo = limpiar(ws.cell(row, col_archivo).value)
        if not archivo:
            continue

        # Primero intenta leer CUMPLE si existe
        if col_cumple:
            estado = estado_ok_no(ws.cell(row, col_cumple).value)
            if estado:
                resultados[archivo] = estado
                continue

        # Si CUMPLE está vacío, calcula con las columnas clave
        valores = []

        for col in [col_resp, col_firmas, col_fecha]:
            if col:
                val = estado_ok_no(ws.cell(row, col).value)
                if val:
                    valores.append(val)

        if valores:
            resultados[archivo] = "NO" if "NO" in valores else "OK"
        else:
            resultados[archivo] = ""

    return resultados


def actualizar_general(wb, estados_prof):
    if "GENERAL" not in wb.sheetnames:
        return

    ws = wb["GENERAL"]

    col_archivo = buscar_columna(ws, ["ARCHIVO"])
    col_prof = None

    for cell in ws[1]:
        h = normalizar(cell.value)
        if "VALIDACION" in h and "PROFESION" in h:
            col_prof = cell.column
            break

    if not col_archivo or not col_prof:
        return

    fill_ok = PatternFill("solid", fgColor="C6EFCE")
    fill_no = PatternFill("solid", fgColor="FFC7CE")

    for row in range(2, ws.max_row + 1):
        archivo = limpiar(ws.cell(row, col_archivo).value)
        if not archivo:
            continue

        estado = estados_prof.get(archivo, "")
        ws.cell(row, col_prof).value = estado

        if estado == "OK":
            ws.cell(row, col_prof).fill = fill_ok
        elif estado == "NO":
            ws.cell(row, col_prof).fill = fill_no


def escapar_textos_tipo_formula(wb):
    """
    Evita que Excel repare el archivo.
    Si un texto OCR empieza con =, +, - o @, Excel puede interpretarlo como fórmula.
    Lo convertimos en texto seguro.
    """
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                val = cell.value

                if val is None:
                    continue

                # Fórmulas generadas accidentalmente por OCR
                if cell.data_type == "f":
                    cell.value = "'" + str(val)
                    continue

                if isinstance(val, str):
                    txt = val.strip()
                    if txt.startswith(("=", "+", "-", "@")):
                        cell.value = "'" + val


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--excel", default="outputs/Reporte_validacion.xlsx")
    args = parser.parse_args()

    excel_path = Path(args.excel)

    wb = load_workbook(excel_path)

    estados_prof = calcular_validacion_profesional(wb)
    actualizar_general(wb, estados_prof)
    escapar_textos_tipo_formula(wb)

    wb.save(excel_path)

    print("Listo amiga.")
    print("Se corrigió:")
    print("- VALIDACION_PROFESIONAL en GENERAL")
    print("- Textos OCR que Excel interpretaba como fórmulas")


if __name__ == "__main__":
    main()

Overwriting src/fix_excel_final.py


Descarga

In [16]:
%cd /content/ramec

!python src/reporte_final.py \
  --excel outputs/Reporte_validacion.xlsx \
  --entidad-esperada "PROINVERSIÓN"

!python src/fix_excel_final.py \
  --excel outputs/Reporte_validacion.xlsx

import pandas as pd
from google.colab import files

REPORTE = "outputs/Reporte_validacion.xlsx"

xls = pd.ExcelFile(REPORTE)
print("Hojas:", xls.sheet_names)

print("\nGENERAL:")
display(pd.read_excel(xls, "GENERAL"))

print("\nVALIDACION_PAGINAS:")
display(pd.read_excel(xls, "VALIDACION_PAGINAS"))

print("\nVALIDACION_LOGOS:")
display(pd.read_excel(xls, "VALIDACION_LOGOS"))

files.download(REPORTE)

/content/ramec
Listo. Excel final generado:
- GENERAL
- VALIDACION_PAGINAS
- VALIDACION_LOGOS
Listo amiga.
Se corrigió:
- VALIDACION_PROFESIONAL en GENERAL
- Textos OCR que Excel interpretaba como fórmulas
Hojas: ['GENERAL', 'ESTANDAR NOMENCLATURA', 'COMPATIBILIDAD_DOCUMENTO', 'COHERENCIA_DOCUMENTO', 'CONTROL_CAMBIOS_DOC', 'VALIDACION_PROFESIONAL', 'VALIDACION_PAGINAS', 'VALIDACION_LOGOS', 'VALIDACION_FINAL_PAGINAS', 'DETALLE_SELLOS_PAGINAS', 'DETALLE_LOGOS_PAGINAS']

GENERAL:


,Ruta,Archivo,ESTANDAR NOMENCLATURA,COMPATIBILIDAD_DOCUMENTO,COHERENCIA_DOCUMENTO,CONTROL_CAMBIOS_DOC,VALIDACION_PROFESIONAL,VALIDACION_PAGINAS,VALIDACION_LOGOS
0,pdfs/AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,OK,OK,NO,OK,OK,OK,NO



VALIDACION_PAGINAS:


,Archivo,Sello_detectado_en_contenido,%_páginas_con_sello,Cantidad_páginas_con_sello,Firmantes_detectados,Cargos_detectados,Responsables_hoja_de_control,Responsables_detectados,Responsables_no_detectados,Firmantes_coinciden,CIP_detectados,CUMPLE,Observación_general
0,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,Sí,100,8,Miguel Núñez Gutiérrez | Eugenia Mari | Armand...,Gerente de Construcción | Gerente de Proyecto,Vicente Nicolas Padilla Aycho | Eugenia Marina...,Vicente Nicolas Padilla Aycho | Eugenia Marina...,NaN,OK,264301,OK,Se detectaron sellos laterales en todas las pá...



VALIDACION_LOGOS:


,Archivo,Logo_detectado_documento,%_páginas_con_logo,Entidad_esperada,Entidades_detectadas,Concedente_OK,CUMPLE,Observación_general
0,AVP-SAV-3000-D-SPC-FPV-ING-000001.pdf,Sí,100,PROINVERSIÓN,MTC | Sociedad Concesionaria Anillo Vial,NO,NO,Se detectó logo en todas las páginas evaluadas...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>